<a href="https://colab.research.google.com/github/polaganirashmitha627-source/PulseVision-AI1/blob/main/Copy_of_PulseVisionAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers gradio torch torchvision pillow reportlab -q

In [ ]:
from transformers import pipeline
from PIL import Image
import gradio as gr

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib import styles

In [ ]:
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
def explain_result(label, symptoms):

    symptoms = symptoms.lower()

    explanations = {

        "NORMAL": {
            "risk": "LOW",
            "tips": "Maintain healthy lifestyle."
        },

        "HEART_DISEASE": {
            "risk": "HIGH",
            "tips": "Consult cardiologist immediately."
        },

        "ARRHYTHMIA": {
            "risk": "MEDIUM",
            "tips": "Monitor heartbeat regularly."
        }
    }

    result = explanations.get(label)

    advice = ""

    if "chest pain" in symptoms:
        advice += "• Chest pain detected\n"

    if "dizziness" in symptoms:
        advice += "• Dizziness noted\n"

    if advice == "":
        advice = "• No major symptom match"

    return result, advice

In [ ]:
def create_pdf_report(name, age, disease, score, risk, symptoms, tips, advice, doctor):

    filename = "PulseVision_Report.pdf"

    doc = SimpleDocTemplate(filename)
    style = styles.getSampleStyleSheet()

    content = []

    content.append(Paragraph("PulseVision AI Health Report", style['Title']))
    content.append(Spacer(1, 10))

    content.append(Paragraph(f"Patient Name: {name}", style['BodyText']))
    content.append(Paragraph(f"Age: {age}", style['BodyText']))
    content.append(Paragraph(f"Prediction: {disease}", style['BodyText']))
    content.append(Paragraph(f"Confidence: {round(score*100,2)} %", style['BodyText']))
    content.append(Paragraph(f"Risk Level: {risk}", style['BodyText']))
    content.append(Paragraph(f"Symptoms: {symptoms}", style['BodyText']))
    content.append(Paragraph(f"Doctor: {doctor}", style['BodyText']))
    content.append(Paragraph(f"Suggestions: {tips}", style['BodyText']))
    content.append(Paragraph(f"Advice: {advice}", style['BodyText']))

    content.append(Spacer(1, 10))
    content.append(Paragraph("Demo Project Only", style['BodyText']))

    doc.build(content)

    return filename

In [ ]:
def predict_health(name, age, image, symptoms):

    img = Image.fromarray(image)

    if img.size[0] < 100:
        return "Invalid Image", None, image

    results = classifier(img)

    score = results[0]['score']

    if score > 0.6:
        disease = "HEART_DISEASE"
        doctor = "Cardiologist"

    elif score > 0.4:
        disease = "ARRHYTHMIA"
        doctor = "Heart Specialist"

    else:
        disease = "NORMAL"
        doctor = "General Physician"

    result, advice = explain_result(disease, symptoms)

    if result['risk'] == "HIGH":
        warning = "Immediate attention required!"

    elif result['risk'] == "MEDIUM":
        warning = "Consult doctor soon"

    else:
        warning = "No major risk"

    pdf = create_pdf_report(
        name, age, disease, score,
        result['risk'], symptoms,
        result['tips'], advice, doctor
    )

    output = f"""
🏥 PULSEVISION AI REPORT

Name: {name}
Age: {age}

Prediction: {disease}
Confidence: {round(score*100,2)} %

Risk: {result['risk']}
Alert: {warning}

Doctor: {doctor}

Symptoms: {symptoms}

Suggestions:
{result['tips']}

Advice:
{advice}
"""

    return output, pdf, image

In [ ]:
interface = gr.Interface(

    fn=predict_health,

    inputs=[
        gr.Textbox(label="Patient Name"),
        gr.Number(label="Age"),
        gr.Image(type="numpy", label="Upload Image"),
        gr.Textbox(label="Symptoms")
    ],

    outputs=[
        gr.Textbox(label="Report", lines=20),
        gr.File(label="Download PDF"),
        gr.Image(label="Preview")
    ],

    title="🏥 PulseVision AI",

    description="AI Medical Assistant with Image + Symptoms + PDF Report",

    theme="soft"
)

In [ ]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://926e2310a3472693f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
